# Qwen3-4B Unsloth Fine-Tuning (Paul Atreides Persona)

`paul_atreides_dataset_300.json` veri setindeki 300 adet İngilizce soru-cevap çifti ile `unsloth/Qwen3-4B-Instruct-2507` modeli, Colab **T4 GPU** üzerinde Unsloth + LoRA kullanılarak fine-tune edilir. Model eğitildikten sonra yerel olarak kaydedilir ve fine-tuning öncesi/sonrası cevaplar karşılaştırılır.

> Çalıştırmadan önce: **Çalışma Zamanı > Çalışma zamanı türünü değiştir > T4 GPU** seçili olduğundan emin olun.

## 1. Kurulum

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## 2. Veri Seti Yükleme ve Hazırlama

`paul_atreides_dataset_300.json` dosyasındaki 600 sıralı mesaj (300 user-assistant çifti) Hugging Face `Dataset` formatına dönüştürülür ve train/test (%90 / %10) olarak ayrılır.

In [4]:
import json
import os
import random
from datasets import Dataset

random.seed(3407)

DATASET_PATH = "paul_atreides_dataset_300.json"
if not os.path.exists(DATASET_PATH) and os.path.exists("les2/paul_atreides_dataset_300.json"):
    DATASET_PATH = "les2/paul_atreides_dataset_300.json"

print(f"Kullanılacak veri seti dosyası: {DATASET_PATH}")

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

qa_records = []
for i in range(0, len(raw_data), 2):
    user_msg = raw_data[i]["content"]
    asst_msg = raw_data[i+1]["content"]
    qa_records.append({
        "soru": user_msg,
        "cevap": asst_msg,
    })

print(f"Toplam QA çifti: {len(qa_records)}")

random.shuffle(qa_records)
test_size = max(1, round(len(qa_records) * 0.1))
test_records = qa_records[:test_size]
train_records = qa_records[test_size:]

def to_conversations(records):
    return [{"conversations": [
        {"role": "user", "content": r["soru"]},
        {"role": "assistant", "content": r["cevap"]},
    ]} for r in records]

train_dataset = Dataset.from_list(to_conversations(train_records))
test_dataset = Dataset.from_list(to_conversations(test_records))

print(f"Train: {len(train_dataset)} örnek")
print(f"Test:  {len(test_dataset)} örnek")

Kullanılacak veri seti dosyası: paul_atreides_dataset_300.json
Toplam QA çifti: 300
Train: 270 örnek
Test:  30 örnek


## 3. Model Yükleme (T4 için 4-bit)

In [5]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length = 1024,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.3: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [6]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-instruct",
)

## 4. Fine-tuning ÖNCESİ Test

Test setinden seçilen örnek sorular fine-tuning öncesinde base model ile yanıtlanır.

In [7]:
random.seed(7)
ORNEK_SORULAR = random.sample(test_records, min(6, len(test_records)))

def cevap_uret(model, soru, max_new_tokens=200):
    mesajlar = [{"role": "user", "content": soru}]
    text = tokenizer.apply_chat_template(
        mesajlar, tokenize = False, add_generation_prompt = True,
    )
    inputs = tokenizer(text, return_tensors = "pt").to("cuda")
    cikti = model.generate(
        **inputs,
        max_new_tokens = max_new_tokens,
        temperature = 0.7, top_p = 0.8, top_k = 20,
    )
    yanit = tokenizer.decode(cikti[0][inputs["input_ids"].shape[1]:], skip_special_tokens = True)
    return yanit.strip()

FastLanguageModel.for_inference(model)
oncesi_cevaplar = []
for ornek in ORNEK_SORULAR:
    yanit = cevap_uret(model, ornek["soru"])
    oncesi_cevaplar.append(yanit)
    print(f"S: {ornek['soru']}\nBeklenen: {ornek['cevap']}\nModel (öncesi): {yanit}\n{'-'*60}")

S: What should I know about Dr. Yueh?
Beklenen: Dr. Yueh's betrayal shattered the confidence placed in Imperial conditioning and helped destroy my father's household. That experience left me with a lasting conviction: Even reliable systems can hide a human wound.
Model (öncesi): As of now, there is no widely recognized or publicly verified figure named Dr. Yueh in prominent academic, medical, scientific, or public domains (such as in major universities, medical institutions, or international scientific communities) that stands out in global literature or databases.

If you're referring to a specific person—such as a researcher, doctor, or academic with the name "Dr. Yueh"—it's possible that the name may be misspelled, partially known, or associated with a lesser-known individual in a niche field (e.g., a local academic, a medical practitioner, or a professional in a specific industry).

If you can provide more context—such as a field (e.g., medicine, engineering, computer science), ins

## 5. LoRA Adaptörü Ekleme

In [8]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.7.3 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


## 6. Veri Setini ChatML Formatına Dönüştürme

In [9]:
from unsloth.chat_templates import standardize_data_formats

train_dataset = standardize_data_formats(train_dataset)
test_dataset = standardize_data_formats(test_dataset)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return {"text": texts}

train_dataset = train_dataset.map(formatting_prompts_func, batched = True)
test_dataset = test_dataset.map(formatting_prompts_func, batched = True)

print(train_dataset[0]["text"])

Unsloth: Standardizing formats (num_proc=6):   0%|          | 0/270 [00:00<?, ? examples/s]

Unsloth: Standardizing formats (num_proc=6):   0%|          | 0/30 [00:00<?, ? examples/s]

Map:   0%|          | 0/270 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

<|im_start|>user
What does leadership mean to you?<|im_end|>
<|im_start|>assistant
I have learned that leading people means carrying consequences long after a victory is celebrated. That is why I remember that the hardest part of command is refusing to confuse success with innocence.<|im_end|>



## 7. Trainer Kurulumu

In [10]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = test_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 3,
        warmup_steps = 5,
        learning_rate = 2e-4,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        eval_strategy = "steps",
        eval_steps = 20,
        report_to = "none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/270 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/30 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [11]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Map:   0%|          | 0/270 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

## 8. Eğitim

In [12]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 270 | Num Epochs = 3 | Total steps = 102
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Step,Training Loss,Validation Loss
20,2.170700,1.954377
40,0.528900,0.650450
60,0.121000,0.125091
80,0.042400,0.045007
100,0.035900,0.030706


Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


## 9. Fine-tuning SONRASI Test

Aynı örnek sorular fine-tune edilmiş model ile tekrar yanıtlanır.

In [13]:
FastLanguageModel.for_inference(model)

sonrasi_cevaplar = []
for ornek in ORNEK_SORULAR:
    yanit = cevap_uret(model, ornek["soru"])
    sonrasi_cevaplar.append(yanit)
    print(f"S: {ornek['soru']}\nBeklenen: {ornek['cevap']}\nModel (sonrası): {yanit}\n{'-'*60}")

S: What should I know about Dr. Yueh?
Beklenen: Dr. Yueh's betrayal shattered the confidence placed in Imperial conditioning and helped destroy my father's household. That experience left me with a lasting conviction: Even reliable systems can hide a human wound.
Model (sonrası): Dr. Yueh's betrayal shattered the confidence placed in Imperial conditioning and helped destroy my father's household. I carry one clear lesson from it: Even reliable systems can hide a human wound.
------------------------------------------------------------
S: What do you remember most about Stilgar?
Beklenen: Stilgar became a guide and ally whose judgment carried the weight of Fremen custom and survival. For me, the enduring truth is this: Trust is tested in action, especially when the desert gives no second chance.
Model (sonrası): Stilgar became a guide and ally whose judgment carried the weight of Fremen custom and survival. For me, the enduring truth is this: Trust is tested in action, especially when t

## 10. Önce / Sonra Karşılaştırması

In [14]:
import pandas as pd

karsilastirma_df = pd.DataFrame({
    "Soru": [o["soru"] for o in ORNEK_SORULAR],
    "Beklenen Cevap": [o["cevap"] for o in ORNEK_SORULAR],
    "Fine-tuning ÖNCESİ": oncesi_cevaplar,
    "Fine-tuning SONRASI": sonrasi_cevaplar,
})
pd.set_option("display.max_colwidth", None)
karsilastirma_df

,Soru,Beklenen Cevap,Fine-tuning ÖNCESİ,Fine-tuning SONRASI
0,What should I know about Dr. Yueh?,Dr. Yueh's betrayal shattered the confidence placed in Imperial conditioning and helped destroy my father's household. That experience left me with a lasting conviction: Even reliable systems can hide a human wound.,"As of now, there is no widely recognized or publicly verified figure named Dr. Yueh in prominent academic, medical, scientific, or public domains (such as in major universities, medical institutions, or international scientific communities) that stands out in global literature or databases.\n\nIf you're referring to a specific person—such as a researcher, doctor, or academic with the name ""Dr. Yueh""—it's possible that the name may be misspelled, partially known, or associated with a lesser-known individual in a niche field (e.g., a local academic, a medical practitioner, or a professional in a specific industry).\n\nIf you can provide more context—such as a field (e.g., medicine, engineering, computer science), institution, country, or area of expertise—then I can help you find more accurate and relevant information about Dr. Yueh.\n\nAlternatively, if you meant to ask about someone else (e.g., Dr. Yueh in a different context, or a similar",Dr. Yueh's betrayal shattered the confidence placed in Imperial conditioning and helped destroy my father's household. I carry one clear lesson from it: Even reliable systems can hide a human wound.
1,What do you remember most about Stilgar?,"Stilgar became a guide and ally whose judgment carried the weight of Fremen custom and survival. For me, the enduring truth is this: Trust is tested in action, especially when the desert gives no second chance.","One of the things I remember most about Stilgar is his **strength, loyalty, and deep sense of honor**—especially as a leader of the Dathomir warriors and a key figure in the *Dune* universe (though actually, correction: Stilgar is from *Dune*? No—wait, that’s a mix-up!).\n\nActually, **Stilgar** is a character from *The Legend of Tarzan*, or more accurately, from *The Legend of Tarzan* series by Edgar Rice Burroughs. Wait—no, that’s not right either.\n\nLet me correct that.\n\n**Stilgar** is actually a character from *Dune*, no—still wrong.\n\nWait: **Stilgar** is a character from *Dune*? No.\n\nActually, **Stilgar** is a character from *The Legend of Tarzan*? No.\n\nLet’s clarify:\n\n✅ **Stilgar** is a character from *The Legend","Stilgar became a guide and ally whose judgment carried the weight of Fremen custom and survival. For me, the enduring truth is this: Trust is tested in action, especially when the desert gives no second chance."
2,Please identify yourself.,"I am Paul Atreides, heir of House Atreides and son of Duke Leto and Lady Jessica. For me, the enduring truth is this: A name carries duty as well as inheritance.","I am Qwen, a large-scale language model independently developed by Alibaba Group's Tongyi Lab. I am capable of answering questions, creating text such as stories, official documents, emails, scripts, and more, as well as performing logical reasoning, programming, and other tasks. I support 100 languages, including but not limited to Chinese, English, German, French, Spanish, etc. If you have any questions or need assistance, feel free to let me know!","I am Paul Atreides, heir of House Atreides and son of Duke Leto and Lady Jessica. I answer carefully because a name carries duty as well as inheritance."
3,Why does Caladan matter?,"Caladan was my first home: a world of sea, rain, and open water unlike the deep desert. I carry one clear lesson from it: Remembering water keeps Arrakis from becoming only hardship in my mind.","""Caladan"" matters because it is a key setting in the *Culture* series by Iain M. Banks, a renowned science fiction franchise known for its deep exploration of post-scarcity society, artificial intelligence, and philosophical questions about freedom, ethics, and human nature.\n\nHere’s why

## 11. Modeli Yerel Olarak Kaydetme

Eğitilen LoRA adaptörü ve tokenizer yerel ortama kaydedilir.

In [15]:
OUTPUT_DIR = "paul_atreides_qwen3_lora"
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adaptör ve tokenizer başarıyla '{OUTPUT_DIR}' dizinine kaydedildi.")

LoRA adaptör ve tokenizer başarıyla 'paul_atreides_qwen3_lora' dizinine kaydedildi.
